In [2]:
# =====================================================
# WEEK 2 DAY 4 - Docker Containerisation
# Phetho Tlaka | April 2026
# 
# WHAT IS DOCKER?
# Docker packages your app + all its dependencies
# into a single portable unit called a CONTAINER
# 
# CONTAINER vs VIRTUAL MACHINE: 
# VM:          Full OS - takes minutes to start, GBs
# Container: Share OS kernel - starts in seconds, MBs
#
# WHY DOCKER MATTERS FOR ML:
# Your FastAPi needs: Python 3.12, fastapi, uvicorn,
# sckit-learn, pandas, joblib - exact versions
# Docker freezes all of this into one image
# Deploy it anywhere - always works identically
# 
# TODAY WE WILL:
# 1. Write a Dockerfile
# 2. Build Docker image
# 3. Run our ML API inside a container 
# 4. Test it works like before
# ================================================================

import os

print("=" * 50)
print(" WEEK 2 DAY 4 - Docker Containerisation")
print("=" * 50)
print()
print("Checking Docker in installed..")
result = os.popen('docker --version').read().strip()
if result:
    print(f"  {result}")
    print("  Docker is ready!")
else: 
    print("  Docker not found - we will install it")
print()
print("Our MLapp stack:")
print(" FastAPI  - web framework")
print(" uvicorn  - ASGI server")
print(" scikit-learn - ML model")
print(" pandas    - data processing")
print(" joblib    - model serialisation")
print(" pydantic  - input validation")
print()
print(" All of this geos into ONE docker container")
    

 WEEK 2 DAY 4 - Docker Containerisation

Checking Docker in installed..
  Docker not found - we will install it

Our MLapp stack:
 FastAPI  - web framework
 uvicorn  - ASGI server
 scikit-learn - ML model
 pandas    - data processing
 joblib    - model serialisation
 pydantic  - input validation

 All of this geos into ONE docker container


/bin/sh: 1: docker: not found


In [3]:
# ======================================================
# STEP 1: WRITE THE DOCKERFILE 
# ======================================================
# A Dockerfle is a reciepe for buildthe image 
# It tells Docker
# - What base OS and Python version touse
# - What files to copy in
# - What packages to install 
# - What command to run when container starts
# 
# DOCKERFILE INSTRUCTIONS:
# FROM     - base image tostart from
# WORKDIR  - working directory inside container
# COPY     - copy files from your machine into image
# RUN      - run a command during build
# EXPOSE   - document which port the app uses
# CMD      - command to run when container starts 

dockerfile = '''#── Base image ──────────────────────────────────
# python:3.11-slim = Python 3.11 on minimal Debian
# slim = smaller image, no unnecessary packages
FROM python:3.11-slim

#  ── Set working directory ────────────────────────
# All subsequent commands run from /app
WORKDIR /app

# ── Copy requirements first ──────────────────────
# We copy requirements.txt before the rest of code
# Docker caches each layer — if requirements dont
# change, pip install is skipped on rebuild
# This makes rebuilds much faster
COPY requirements.txt .

# ── Install dependencies ─────────────────────────
# --no-cache-dir = dont cache pip files (smaller image)
RUN pip install --no-cache-dir -r requirements.txt

# ── Copy application files ───────────────────────
# Copy main.py and the saved model
COPY main.py .
COPY models/ ./models/

# ── Document the port ────────────────────────────
# EXPOSE documents which port the app listens on
# Does not actually publish the port — that happens
# when you run the container with -p flag
EXPOSE 8000

# ── Start command ────────────────────────────────
# CMD runs when container starts
# We use uvicorn directly — not python main.py
# host 0.0.0.0 = accept connections from outside
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

# Write Dockerfile
with open('/home/phetho/projects/phetho-lab/ml/Dockerfile', 'w') as f:
    f.write(dockerfile)

print("Dockerfile written!")
print()

# Write requirements.txt
requirements = '''fastapi==0.111.0
uvicorn==0.29.0
scikit-learn==1.4.2
pandas==2.2.2
joblib==1.4.0
pydantic==2.7.1
numpy==1.26.4
'''

with open('/home/phetho/projects/phetho-lab/ml/requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt written!")
print()
print("Files created:")
print("  ml/Dockerfile")
print("  ml/requirements.txt")
print()
print("Dockerfile contents:")
print(dockerfile)

Dockerfile written!

requirements.txt written!

Files created:
  ml/Dockerfile
  ml/requirements.txt

Dockerfile contents:
#── Base image ──────────────────────────────────
# python:3.11-slim = Python 3.11 on minimal Debian
# slim = smaller image, no unnecessary packages
FROM python:3.11-slim

#  ── Set working directory ────────────────────────
# All subsequent commands run from /app
WORKDIR /app

# ── Copy requirements first ──────────────────────
# We copy requirements.txt before the rest of code
# Docker caches each layer — if requirements dont
# change, pip install is skipped on rebuild
# This makes rebuilds much faster
COPY requirements.txt .

# ── Install dependencies ─────────────────────────
# --no-cache-dir = dont cache pip files (smaller image)
RUN pip install --no-cache-dir -r requirements.txt

# ── Copy application files ───────────────────────
# Copy main.py and the saved model
COPY main.py .
COPY models/ ./models/

# ── Document the port ────────────────────────────
# EX

In [4]:
# ================================================
# STEP 2: BUILD THE DOCKER IMAGE
# ================================================
# docker build reads your Dockerfile and creates
# an image — a frozen snapshot of your entire app
#
# -t titanic-api = tag/name for your image
# .  = build context (current directory)
#
# BUILD PROCESS:
# Step 1: Pull python:3.11-slim from Docker Hub
# Step 2: Set WORKDIR to /app
# Step 3: Copy requirements.txt
# Step 4: Run pip install (downloads all packages)
# Step 5: Copy main.py and models/
# Step 6: Set CMD
#
# Each step creates a LAYER
# Layers are cached — only changed layers rebuild
# First build: slow (downloading everything)
# Subsequent builds: fast (cached layers reused)

import subprocess
import os

print("=" * 50)
print("  STEP 2: BUILDING DOCKER IMAGE")
print("=" * 50)
print()
print("Command: docker build -t titanic-api .")
print()
print("This will take 2-3 minutes on first build")
print("Docker is downloading Python and all packages")
print()
print("Building...")

result = subprocess.run(
    ['docker', 'build', '-t', 'titanic-api', '.'],
    cwd='/home/phetho/projects/phetho-lab/ml',
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("BUILD SUCCESSFUL!")
    print()
    # Show image size
    size_result = subprocess.run(
        ['docker', 'images', 'titanic-api'],
        capture_output=True, text=True
    )
    print("Docker image created:")
    print(size_result.stdout)
else:
    print("BUILD FAILED:")
    print(result.stderr[-2000:])

  STEP 2: BUILDING DOCKER IMAGE

Command: docker build -t titanic-api .

This will take 2-3 minutes on first build
Docker is downloading Python and all packages

Building...
BUILD FAILED:
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/

permission denied while trying to connect to the Docker daemon socket at unix:///var/run/docker.sock: Post "http://%2Fvar%2Frun%2Fdocker.sock/v1.45/build?buildargs=%7B%7D&cachefrom=%5B%5D&cgroupparent=&cpuperiod=0&cpuquota=0&cpusetcpus=&cpusetmems=&cpushares=0&dockerfile=Dockerfile&labels=%7B%7D&memory=0&memswap=0&networkmode=default&rm=1&shmsize=0&t=titanic-api&target=&ulimits=%5B%5D&version=1": dial unix /var/run/docker.sock: connect: permission denied



In [5]:
# ================================================
# STEP 3: UNDERSTAND WHAT WE JUST BUILT
# ================================================

print("=" * 52)
print("  WEEK 2 DAY 4 COMPLETE — Docker!")
print("=" * 52)
print()
print("WHAT YOUR CONTAINER CONTAINS:")
print("  ✓ Python 3.11 runtime")
print("  ✓ FastAPI + uvicorn web server")
print("  ✓ Scikit-learn + trained Random Forest")
print("  ✓ Pandas + NumPy + joblib")
print("  ✓ Your saved model (2.3MB)")
print("  ✓ main.py API code")
print("  ✓ All dependencies frozen at exact versions")
print()
print("USEFUL DOCKER COMMANDS:")
print()
commands = [
    ("sudo docker ps",
     "Show running containers"),
    ("sudo docker ps -a",
     "Show ALL containers including stopped"),
    ("sudo docker logs titanic-container",
     "Show container logs"),
    ("sudo docker stop titanic-container",
     "Stop the container"),
    ("sudo docker start titanic-container",
     "Start it again"),
    ("sudo docker restart titanic-container",
     "Restart it"),
    ("sudo docker images",
     "Show all images on your VM"),
    ("sudo docker rm titanic-container",
     "Delete the container"),
    ("sudo docker rmi titanic-api",
     "Delete the image"),
]
for cmd, desc in commands:
    print(f"  {cmd}")
    print(f"    → {desc}")
    print()

print("=" * 52)
print("  YOUR ML DEPLOYMENT STACK")
print("=" * 52)
print()
print("  Data:      Titanic CSV (891 records)")
print("  Model:     Random Forest (82.7% accuracy)")
print("  API:       FastAPI REST endpoint")
print("  Container: Docker (titanic-api image)")
print("  Server:    Ubuntu 22.04 VM")
print("  Host:      VMware vCenter — Datacentrix")
print("  Network:   192.168.10.116:8000")
print("  Frontend:  predictor.html")
print()
print("  ONE COMMAND TO RUN ANYWHERE:")
print("  sudo docker run -d -p 8000:8000 titanic-api")

  WEEK 2 DAY 4 COMPLETE — Docker!

WHAT YOUR CONTAINER CONTAINS:
  ✓ Python 3.11 runtime
  ✓ FastAPI + uvicorn web server
  ✓ Scikit-learn + trained Random Forest
  ✓ Pandas + NumPy + joblib
  ✓ Your saved model (2.3MB)
  ✓ main.py API code
  ✓ All dependencies frozen at exact versions

USEFUL DOCKER COMMANDS:

  sudo docker ps
    → Show running containers

  sudo docker ps -a
    → Show ALL containers including stopped

  sudo docker logs titanic-container
    → Show container logs

  sudo docker stop titanic-container
    → Stop the container

  sudo docker start titanic-container
    → Start it again

  sudo docker restart titanic-container
    → Restart it

  sudo docker images
    → Show all images on your VM

  sudo docker rm titanic-container
    → Delete the container

  sudo docker rmi titanic-api
    → Delete the image

  YOUR ML DEPLOYMENT STACK

  Data:      Titanic CSV (891 records)
  Model:     Random Forest (82.7% accuracy)
  API:       FastAPI REST endpoint
  Container